# 🚪 Türlisten-Automatisierung: BIM Door Validation Tool

*Gruppe A | Data Management HS 2024/25*

In [1]:
%pip install ifcopenshell ipywidgets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 14.4 MB/s eta 0:00:00


In [2]:
# ============================================================================
# IMPORTS AND SETUP
# ============================================================================
from __future__ import annotations

import csv
import json
import re
import os
import warnings
from pathlib import Path
from typing import Any, Optional
from dataclasses import dataclass, field
from enum import Enum
from datetime import datetime

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

warnings.filterwarnings('ignore', category=FutureWarning)

try:
    import ifcopenshell
    from ifcopenshell.util.element import get_psets
    IFC_AVAILABLE = True
except ImportError:
    IFC_AVAILABLE = False
    print("⚠️ ifcopenshell not installed. Run the install cell above and restart runtime.")

OUTPUT_CSV = "door_check_report.csv"
OUTPUT_HTML = "door_ampel_dashboard.html"

GITHUB_REPO_OWNER = 'louistrue'
GITHUB_REPO_NAME = 'learn-ifc-bfh25-D'
GITHUB_FOLDER_PATH = 'Modelle/BFH-25'

doors_data = None

print("✓ Setup complete")

✓ Setup complete


In [3]:
# ============================================================================
# SAMPLE DATA
# ============================================================================

SAMPLE_DOORS = [
    {"id": "Door-001", "name": "Eingang Nord", "OverallWidth": 1000, "OverallHeight": 2100,
     "psets": {"Pset_DoorCommon": {"FireRating": "EI60"}}, "room": "Lobby", "storey": "EG"},
    {"id": "Door-002", "name": "WC Damen", "OverallWidth": 780, "OverallHeight": 2100,
     "psets": {"Pset_DoorCommon": {}}, "room": "Sanitär", "storey": "EG"},
    {"id": "Door-003", "name": "Lager", "OverallWidth": 600, "OverallHeight": 2000,
     "psets": {"Pset_DoorCommon": {"FireRating": None}}, "room": "Lager", "storey": "UG"},
    {"id": "Door-004", "name": "Schranktür", "OverallWidth": 250, "OverallHeight": 1800,
     "psets": {"Pset_DoorCommon": {}}, "room": "Akten", "storey": "EG"},
    {"id": "Door-005", "name": "Fluchttür Ost", "OverallWidth": 1200, "OverallHeight": 2200,
     "psets": {"Pset_DoorCommon": {"FireRating": "EI 90"}}, "room": "Treppenhaus", "storey": "EG"}
]

print(f"✓ Sample data: {len(SAMPLE_DOORS)} doors")

✓ Sample data: 5 doors


In [4]:
# ============================================================================
# IFC PARSING - FIXED VERSION
# ============================================================================

def normalize_to_mm(value: Any) -> tuple[Optional[int], str]:
    """Normalize dimension to millimeters. Returns (value_mm, unit_detected)."""
    if value is None:
        return None, "none"

    if isinstance(value, str):
        s = value.strip().lower()
        if s.endswith("mm"):
            try:
                return int(round(float(s[:-2].strip().replace(",", ".")))), "mm"
            except ValueError:
                return None, "parse_error"
        if s.endswith("m"):
            try:
                return int(round(float(s[:-1].strip().replace(",", ".")) * 1000)), "m"
            except ValueError:
                return None, "parse_error"
        try:
            value = float(s.replace(",", "."))
        except ValueError:
            return None, "parse_error"

    if isinstance(value, (int, float)):
        if value < 10:
            return int(round(value * 1000)), "m_auto"
        elif value >= 100:
            return int(round(value)), "mm_auto"
        else:
            return int(round(value)), "mm_uncertain"

    return None, "unknown"


def parse_ifc_doors(file_path: str) -> list[dict]:
    """
    Parse doors from IFC. Extracts dimensions from:
    1. Direct IfcDoor attributes
    2. IfcDoorType attributes
    3. Quantity sets (Qto_DoorBaseQuantities)
    4. Other property sets
    """
    if not IFC_AVAILABLE:
        raise ImportError("ifcopenshell required")

    model = ifcopenshell.open(file_path)
    doors = model.by_type("IfcDoor")

    if not doors:
        print(f"⚠️ No IfcDoor elements in {Path(file_path).name}")
        return []

    # Spatial containment
    storey_map, room_map = {}, {}
    for rel in model.by_type('IfcRelContainedInSpatialStructure'):
        structure = rel.RelatingStructure
        if structure:
            for el in rel.RelatedElements:
                if structure.is_a('IfcBuildingStorey'):
                    storey_map[el.GlobalId] = structure.Name or structure.LongName
                elif structure.is_a('IfcSpace'):
                    room_map[el.GlobalId] = structure.Name or structure.LongName

    parsed = []
    for door in doors:
        psets = get_psets(door)

        # Extract dimensions from multiple sources
        width = door.OverallWidth
        height = door.OverallHeight

        # Try door type
        if width is None or height is None:
            try:
                from ifcopenshell.util.element import get_type
                door_type = get_type(door)
                if door_type:
                    width = width or getattr(door_type, 'OverallWidth', None)
                    height = height or getattr(door_type, 'OverallHeight', None)
            except:
                pass

        # Try quantity sets
        if width is None or height is None:
            for qto_name in ['Qto_DoorBaseQuantities', 'BaseQuantities']:
                qto = psets.get(qto_name, {})
                if isinstance(qto, dict):
                    width = width or qto.get('Width') or qto.get('width')
                    height = height or qto.get('Height') or qto.get('height')

        # Try any pset with Width/Height
        if width is None or height is None:
            for pset_data in psets.values():
                if isinstance(pset_data, dict):
                    width = width or pset_data.get('Width') or pset_data.get('OverallWidth')
                    height = height or pset_data.get('Height') or pset_data.get('OverallHeight')

        # Normalize to mm for storage
        width_mm, _ = normalize_to_mm(width)
        height_mm, _ = normalize_to_mm(height)

        parsed.append({
            "id": door.GlobalId,
            "name": door.Name,
            "OverallWidth": width_mm,    # Store normalized mm value!
            "OverallHeight": height_mm,  # Store normalized mm value!
            "psets": psets,
            "room": room_map.get(door.GlobalId),
            "storey": storey_map.get(door.GlobalId)
        })

    return parsed


def load_doors_from_file(file_path: str) -> bool:
    global doors_data
    try:
        doors_data = parse_ifc_doors(file_path)
        if doors_data:
            print(f"✅ Loaded {len(doors_data)} doors")
            # Debug: show first door's dimensions
            d = doors_data[0]
            print(f"   Sample: {d['name']} - Width={d['OverallWidth']}mm, Height={d['OverallHeight']}mm")
            return True
        doors_data = SAMPLE_DOORS
        return True
    except Exception as e:
        print(f"❌ Error: {e}")
        return False


def load_sample_data():
    global doors_data
    doors_data = SAMPLE_DOORS
    print(f"✅ Loaded {len(doors_data)} sample doors")


print("✓ IFC parser ready")

✓ IFC parser ready


In [5]:
# ============================================================================
# LOADING WIDGET
# ============================================================================

def fetch_github_files():
    import urllib.request, json as j
    url = f'https://api.github.com/repos/{GITHUB_REPO_OWNER}/{GITHUB_REPO_NAME}/contents/{GITHUB_FOLDER_PATH}'
    try:
        with urllib.request.urlopen(url, timeout=10) as resp:
            return [f['name'] for f in j.loads(resp.read().decode()) if f['name'].endswith('.ifc')] or ['No IFC files']
    except:
        return ['Error fetching']

def on_load_clicked(b):
    global doors_data
    with output_area:
        clear_output()
        if loading_method.value == 'A' and 'Error' not in github_dropdown.value:
            import urllib.request
            f = github_dropdown.value
            url = f'https://raw.githubusercontent.com/{GITHUB_REPO_OWNER}/{GITHUB_REPO_NAME}/main/{GITHUB_FOLDER_PATH}/{f}'
            print(f"📥 Downloading {f}...")
            urllib.request.urlretrieve(url, f)
            load_doors_from_file(f)
        elif loading_method.value == 'B' and file_upload.value:
            uf = next(iter(file_upload.value.values()))
            fn = uf['metadata']['name']
            with open(fn, 'wb') as f: f.write(uf['content'])
            load_doors_from_file(fn)
        elif loading_method.value == 'C':
            load_sample_data()
        else:
            print("❌ Select a valid option")

        if doors_data:
            print(f"\n✅ Ready! Run validation cells below.")

# Widgets
loading_method = widgets.RadioButtons(
    options=[('A) GitHub', 'A'), ('B) Upload', 'B'), ('C) Sample Data', 'C')],
    value='C', description='Source:'
)
github_dropdown = widgets.Dropdown(options=fetch_github_files(), description='File:')
file_upload = widgets.FileUpload(accept='.ifc', multiple=False)
load_btn = widgets.Button(description='Load Data', button_style='success', icon='download')
load_btn.on_click(on_load_clicked)
output_area = widgets.Output()

display(widgets.VBox([loading_method, github_dropdown, file_upload, load_btn, output_area]))

In [24]:
# ============================================================================
# VALIDATION RULES
# ============================================================================

class Severity(Enum):
    GREEN = "green"
    YELLOW = "yellow"
    RED = "red"
    def __lt__(self, other):
        return {Severity.GREEN: 0, Severity.YELLOW: 1, Severity.RED: 2}[self] < \
               {Severity.GREEN: 0, Severity.YELLOW: 1, Severity.RED: 2}[other]

@dataclass
class ValidationIssue:
    rule_id: str
    severity: Severity
    message: str
    actual_value: Any = None
    expected_value: Any = None
    def to_dict(self):
        return {"rule_id": self.rule_id, "severity": self.severity.value,
                "message": self.message, "actual_value": self.actual_value}

@dataclass
class DoorElement:
    global_id: str
    name: Optional[str] = None
    overall_width_mm: Optional[int] = None
    overall_height_mm: Optional[int] = None
    psets: dict = field(default_factory=dict)
    room_name: Optional[str] = None
    storey: Optional[str] = None

@dataclass
class ValidationResult:
    door: DoorElement
    issues: list = field(default_factory=list)
    traffic_light: Severity = Severity.GREEN
    def compute_traffic_light(self):
        self.traffic_light = max((i.severity for i in self.issues), default=Severity.GREEN)

# Rules - dimensions are now stored in MM in door_dict!
RULES = [
    {"id": "fire_rating_required", "property": ["Pset_DoorCommon", "FireRating"],
     "check": "exists", "severity": "red", "message": "Fehlende FireRating-Angabe"},
    {"id": "fire_rating_format", "property": ["Pset_DoorCommon", "FireRating"],
     "check": "pattern", "pattern": "^(EI|E|EW)\\s*\\d{2,3}(-C\\d?)?$",
     "severity": "yellow", "message": "FireRating-Format ungültig"},
    {"id": "width_barrier_free", "property": ["", "OverallWidth"],
     "check": "min_value", "value": 900, "severity": "yellow",
     "message": "Breite < 900 mm (nicht barrierefrei)"},
    {"id": "width_minimum", "property": ["", "OverallWidth"],
     "check": "min_value", "value": 300, "severity": "red",
     "message": "Breite < 300 mm (unplausibel)"},
    {"id": "height_minimum", "property": ["", "OverallHeight"],
     "check": "min_value", "value": 1800, "severity": "yellow",
     "message": "Höhe < 1800 mm"}
]

print(f"✓ {len(RULES)} rules defined")

✓ 5 rules defined


In [25]:
# ============================================================================
# VALIDATION ENGINE - FIXED
# ============================================================================

def get_property_value(element: dict, property_spec: list[str]) -> Any:
    """Get property from element dict or pset."""
    pset_name, prop_name = property_spec
    if pset_name == "":
        return element.get(prop_name)
    return element.get("psets", {}).get(pset_name, {}).get(prop_name)


def evaluate_rule(rule: dict, element: dict) -> list[ValidationIssue]:
    """Evaluate rule against element. Dimensions are already in mm!"""
    issues = []
    check = rule["check"]
    severity = Severity(rule["severity"])
    value = get_property_value(element, rule["property"])

    if check == "exists":
        if value is None or (isinstance(value, str) and not value.strip()):
            issues.append(ValidationIssue(rule["id"], severity, rule["message"], value))

    elif check == "min_value":
        # Value is already in mm (normalized during parsing)
        threshold = rule["value"]
        if value is not None and value < threshold:
            issues.append(ValidationIssue(
                rule["id"], severity, rule["message"],
                f"{value} mm", f">= {threshold} mm"
            ))

    elif check == "max_value":
        threshold = rule["value"]
        if value is not None and value > threshold:
            issues.append(ValidationIssue(
                rule["id"], severity, rule["message"],
                f"{value} mm", f"<= {threshold} mm"
            ))

    elif check == "pattern":
        if value and isinstance(value, str) and value.strip():
            if not re.match(rule.get("pattern", ".*"), value.strip(), re.IGNORECASE):
                issues.append(ValidationIssue(rule["id"], severity, rule["message"], value))

    return issues


def validate_doors(doors: list[dict], rules: list[dict]) -> list[ValidationResult]:
    """Validate all doors against all rules."""
    results = []
    for d in doors:
        door = DoorElement(
            global_id=d.get("id", "?"),
            name=d.get("name"),
            overall_width_mm=d.get("OverallWidth"),  # Already in mm
            overall_height_mm=d.get("OverallHeight"),
            psets=d.get("psets", {}),
            room_name=d.get("room"),
            storey=d.get("storey")
        )
        all_issues = []
        for rule in rules:
            all_issues.extend(evaluate_rule(rule, d))
        result = ValidationResult(door=door, issues=all_issues)
        result.compute_traffic_light()
        results.append(result)
    return results

print("✓ Validation engine ready")

✓ Validation engine ready


In [26]:
# ============================================================================
# OUTPUT GENERATION
# ============================================================================

def write_csv(results, path):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["GlobalId","Name","Room","Storey","Width_mm","Height_mm","Status","Issues"])
        for r in results:
            w.writerow([r.door.global_id, r.door.name, r.door.room_name, r.door.storey,
                       r.door.overall_width_mm, r.door.overall_height_mm, r.traffic_light.value,
                       json.dumps([i.to_dict() for i in r.issues], ensure_ascii=False)])
    print(f"✓ CSV: {path}")

def generate_html(results, path):
    total, red, yellow, green = len(results), \
        sum(1 for r in results if r.traffic_light == Severity.RED), \
        sum(1 for r in results if r.traffic_light == Severity.YELLOW), \
        sum(1 for r in results if r.traffic_light == Severity.GREEN)

    colors = {"green": "#c6efce", "yellow": "#fff2cc", "red": "#f4c7c3"}

    html = f'''<!DOCTYPE html><html><head><meta charset="utf-8"><title>Tür Dashboard</title>
<style>
body{{font-family:system-ui;margin:20px;background:#f5f5f5}}
.container{{max-width:1400px;margin:0 auto}}
.summary{{display:flex;gap:20px;margin:20px 0;flex-wrap:wrap}}
.card{{background:white;border-radius:8px;padding:20px;min-width:120px;box-shadow:0 2px 4px rgba(0,0,0,0.1);text-align:center}}
.card.red{{border-left:5px solid #dc3545}}.card.yellow{{border-left:5px solid #ffc107}}.card.green{{border-left:5px solid #28a745}}
.num{{font-size:2em;font-weight:bold}}
.filters{{margin:20px 0}}
.btn{{padding:8px 16px;margin-right:10px;border:none;border-radius:4px;cursor:pointer}}
.btn.active{{outline:3px solid #333}}
table{{width:100%;border-collapse:collapse;background:white;box-shadow:0 2px 4px rgba(0,0,0,0.1)}}
th,td{{padding:12px;text-align:left;border-bottom:1px solid #ddd}}
th{{background:#f8f9fa}}
tr.red{{background:{colors["red"]}}}tr.yellow{{background:{colors["yellow"]}}}tr.green{{background:{colors["green"]}}}
.issue{{margin:2px 0;padding:2px 6px;border-radius:3px;font-size:0.85em}}
.issue.red{{background:rgba(220,53,69,0.2)}}.issue.yellow{{background:rgba(255,193,7,0.2)}}
.hidden{{display:none}}
</style></head><body>
<div class="container">
<h1>🚪 Tür Ampel Dashboard</h1>
<div class="summary">
<div class="card"><div class="num">{total}</div><div>Total</div></div>
<div class="card red"><div class="num">{red}</div><div>🔴 Kritisch</div></div>
<div class="card yellow"><div class="num">{yellow}</div><div>🟡 Warnung</div></div>
<div class="card green"><div class="num">{green}</div><div>🟢 OK</div></div>
</div>
<div class="filters">
<button class="btn active" style="background:#e0e0e0" onclick="filter('all')">Alle</button>
<button class="btn" style="background:{colors['red']}" onclick="filter('red')">🔴 Kritisch</button>
<button class="btn" style="background:{colors['yellow']}" onclick="filter('yellow')">🟡 Warnung</button>
<button class="btn" style="background:{colors['green']}" onclick="filter('green')">🟢 OK</button>
</div>
<table><thead><tr><th>ID</th><th>Name</th><th>Raum</th><th>Geschoss</th><th>Breite</th><th>Höhe</th><th>Status</th><th>Issues</th></tr></thead><tbody>
'''

    for r in results:
        s = r.traffic_light.value
        iss = "".join([f'<div class="issue {i.severity.value}">{i.severity.value.upper()}: {i.message}</div>' for i in r.issues]) or "✓"
        html += f'<tr class="{s}" data-s="{s}"><td>{r.door.global_id[:8]}...</td><td>{r.door.name or "-"}</td><td>{r.door.room_name or "-"}</td><td>{r.door.storey or "-"}</td><td>{r.door.overall_width_mm}</td><td>{r.door.overall_height_mm}</td><td><b>{s.upper()}</b></td><td>{iss}</td></tr>\n'

    html += f'''</tbody></table>
<p style="color:#999;margin-top:20px">Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}</p>
</div>
<script>
function filter(s){{document.querySelectorAll('.btn').forEach(b=>b.classList.remove('active'));event.target.classList.add('active');
document.querySelectorAll('tbody tr').forEach(r=>r.classList.toggle('hidden',s!='all'&&r.dataset.s!=s));}}
</script></body></html>'''

    Path(path).write_text(html, encoding="utf-8")
    print(f"✓ HTML: {path}")

print("✓ Output generators ready")

✓ Output generators ready


In [27]:
# ============================================================================
# RUN VALIDATION
# ============================================================================

if doors_data is None:
    print("⚠️ Load data first using the widget above!")
else:
    print("="*50)
    print("🔍 VALIDATING...")
    print("="*50)

    # Debug: Show what rules will check
    print(f"\n📋 Active rules:")
    for r in RULES:
        if r['check'] == 'min_value':
            print(f"   {r['id']}: {r['property'][1]} >= {r['value']}mm ({r['severity']})")

    # Debug: Show sample door values
    print(f"\n📊 Sample door dimensions:")
    for d in doors_data[:3]:
        print(f"   {d['name']}: W={d['OverallWidth']}mm, H={d['OverallHeight']}mm")

    results = validate_doors(doors_data, RULES)

    write_csv(results, OUTPUT_CSV)
    generate_html(results, OUTPUT_HTML)

    # Summary
    print("\n" + "="*50)
    print("📊 RESULTS")
    print("="*50)
    total = len(results)
    red = sum(1 for r in results if r.traffic_light == Severity.RED)
    yellow = sum(1 for r in results if r.traffic_light == Severity.YELLOW)
    green = sum(1 for r in results if r.traffic_light == Severity.GREEN)

    print(f"Total:       {total}")
    print(f"🔴 Kritisch: {red} ({red/total*100:.0f}%)")
    print(f"🟡 Warnung:  {yellow} ({yellow/total*100:.0f}%)")
    print(f"🟢 OK:       {green} ({green/total*100:.0f}%)")

🔍 VALIDATING...

📋 Active rules:
   width_barrier_free: OverallWidth >= 900mm (yellow)
   width_minimum: OverallWidth >= 1200mm (red)
   height_minimum: OverallHeight >= 1800mm (yellow)

📊 Sample door dimensions:
   Type 00.02: W=1010mm, H=2355mm
   Type 00.02: W=1010mm, H=2355mm
   Type 00.02: W=1010mm, H=2355mm
✓ CSV: door_check_report.csv
✓ HTML: door_ampel_dashboard.html

📊 RESULTS
Total:       57
🔴 Kritisch: 41 (72%)
🟡 Warnung:  16 (28%)
🟢 OK:       0 (0%)


In [28]:
# Show doors failing width check
if 'results' in dir():
    print("\n🔴 DOORS WITH WIDTH ISSUES:")
    for r in results:
        width_issues = [i for i in r.issues if 'width' in i.rule_id.lower()]
        if width_issues:
            print(f"  {r.door.name}: {r.door.overall_width_mm}mm")
            for i in width_issues:
                print(f"    → {i.message} (actual: {i.actual_value})")


🔴 DOORS WITH WIDTH ISSUES:
  Type 00.02: 1010mm
    → Breite < 300 mm (unplausibel) (actual: 1010 mm)
  Type 00.02: 1010mm
    → Breite < 300 mm (unplausibel) (actual: 1010 mm)
  Type 00.02: 1010mm
    → Breite < 300 mm (unplausibel) (actual: 1010 mm)
  Type 00.02: 1010mm
    → Breite < 300 mm (unplausibel) (actual: 1010 mm)
  Type 01.02: 1010mm
    → Breite < 300 mm (unplausibel) (actual: 1010 mm)
  Type 01.02: 1010mm
    → Breite < 300 mm (unplausibel) (actual: 1010 mm)
  Type 01.02: 1010mm
    → Breite < 300 mm (unplausibel) (actual: 1010 mm)
  Type 01.02: 1010mm
    → Breite < 300 mm (unplausibel) (actual: 1010 mm)
  Type 01.02: 1010mm
    → Breite < 300 mm (unplausibel) (actual: 1010 mm)
  Type 01.02: 1010mm
    → Breite < 300 mm (unplausibel) (actual: 1010 mm)
  Type 01.02: 1010mm
    → Breite < 300 mm (unplausibel) (actual: 1010 mm)
  Type 01.02: 1010mm
    → Breite < 300 mm (unplausibel) (actual: 1010 mm)
  Type 00.01: 995mm
    → Breite < 300 mm (unplausibel) (actual: 995 mm)

In [30]:
# Download files (Colab)
try:
    from google.colab import files
    files.download(OUTPUT_CSV)
    files.download(OUTPUT_HTML)
except:
    print(f"Files saved: {OUTPUT_CSV}, {OUTPUT_HTML}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>